# Day 6: Advanced Analytics & Risk Metrics
This notebook executes advanced quantitative risk modeling and customer analytics for the Bluestock Mutual Fund portfolio:
1. **Historical Value at Risk (VaR 95%) & Conditional VaR (Expected Shortfall)**
2. **Rolling 90-Day Sharpe Ratio Dynamics Across Market Cycles**
3. **Investor Cohort Analysis (Acquisition Year vs Capital Contribution)**
4. **SIP Continuity & Churn Prediction (At-Risk Investor Segmentation)**
5. **Mutual Fund Recommender Engine (Risk-Profile Driven Scoring)**
6. **Sector Concentration Measurement (Herfindahl-Hirschman Index - HHI)**


In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Connect to database
conn = sqlite3.connect('../db/bluestock_mf.db')
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
conn.close()
print(f'Database verified with {len(tables)} tables: {list(tables["name"])}')


## 1. Historical VaR (95%) and CVaR (Expected Shortfall)
Evaluating 1-day tail risk for all 40 mutual fund schemes at 95% statistical confidence.

In [ ]:
var_df = pd.read_csv('../data/processed/var_cvar_report.csv')
print('Top 10 Highest Risk Schemes (Highest VaR/CVaR):')
var_df[['amfi_code', 'scheme_name', 'category', 'var_95_pct', 'cvar_95_pct', 'worst_day_return_pct']].head(10)


## 2. Rolling 90-Day Sharpe Ratio Across Asset Classes
Visualizing risk-adjusted efficiency trends through market corrections and expansion phases.

In [ ]:
display(Image(filename='../reports/rolling_sharpe_chart.png'))


## 3. Investor Cohort Analysis
Tracking retail investor behavior, average SIP ticket sizes, and asset preferences by onboarding cohort year.

In [ ]:
cohort_df = pd.read_csv('../data/processed/cohort_analysis.csv')
cohort_df


## 4. SIP Continuity & Churn Risk Analysis
Identifying systematic investors with average inter-transaction gaps exceeding 35 days.

In [ ]:
cont_df = pd.read_csv('../data/processed/sip_continuity.csv')
at_risk = cont_df[cont_df['is_at_risk']]
print(f'Total Analyzed SIP Investors (6+ transactions): {len(cont_df)}')
print(f'At-Risk Investors: {len(at_risk)} ({len(at_risk)/len(cont_df)*100:.2f}%)')
at_risk.head(10)


## 5. Mutual Fund Recommendation Engine
Recommending top risk-adjusted mutual funds mapped to investor risk tolerance (`Low`, `Moderate`, `High`).

In [ ]:
import sys
sys.path.append('../scripts')
from recommender import recommend_funds

for risk in ['Low', 'Moderate', 'High']:
    print(f'=== Recommendations for {risk.upper()} Risk Appetite ===')
    display(recommend_funds(risk, top_n=3))
    print()


## 6. Sector Concentration Analysis (HHI)
Herfindahl-Hirschman Index evaluation: flagging funds with HHI > 2500 as highly concentrated.

In [ ]:
hhi_df = pd.read_csv('../data/processed/sector_hhi.csv')
display(Image(filename='../reports/sector_hhi_chart.png'))
print('Funds with Elevated Concentration Risk (HHI > 2500):')
hhi_df[hhi_df['is_concentrated']][['amfi_code', 'scheme_name', 'category', 'top_sector', 'top_sector_weight_pct', 'hhi_score']]


## Key Advanced Analytics Insights (Executive Summary)

1. **Small Cap Tail-Risk Asymmetry**: Small Cap equity schemes carry the highest 1-day 95% Historical VaR (-2.45% to -2.69%) and CVaR (-3.06% to -3.25%), compared to Liquid and Debt funds (VaR between -0.05% and -0.35%).
2. **Sharpe Ratio Cyclicality**: Rolling 90-day Sharpe ratios for equity schemes fluctuate between -2.0 and +4.0 during macro shocks, while Short-Term Debt and Liquid funds maintain a steady, positive Sharpe trajectory (>2.0) across all market regimes.
3. **Cohort Capital Longevity**: The 2024 investor cohort accounts for over 98% of total transactional volume (₹349.11 Crore), exhibiting a stable mean SIP commitment of ₹10,996/month with strong preference for Equity funds.
4. **Critical SIP Churn Vulnerability**: 97.80% of investors with 6+ SIPs experience average payment intervals > 35 days, indicating high mandate bounce rates and an immediate operational need for automated UPI-Autopay nudges.
5. **High Sectoral Concentration in Thematic Funds**: 4 mutual funds exhibit HHI > 2500, led by Axis Bluechip (HHI 2967.69 with 48.69% IT exposure), making their risk profile highly sensitive to single-sector economic cycles.
